## גיבוי ושיתוף: GitHub — `remote`, `clone`, `push`, `pull`

כל מה שעשינו עד עכשיו קרה **מקומית** — ההיסטוריה חיה בתיקיית `.git` על המחשב שלכם בלבד. אם המחשב הזה יאבד, גם ההיסטוריה תאבד. **GitHub** הוא שירות שמארח עותק של הריפו שלכם באינטרנט — **repo מרוחק** (remote), שאפשר להעתיק ממנו ואליו.

ארבע פקודות מקשרות בין הריפו המקומי לריפו המרוחק:

- **`git remote add origin <כתובת>`** — "זוכר" איפה נמצא הריפו המרוחק, תחת שם (בדרך כלל `origin`).
- **`git push`** — שולח קומיטים מהריפו המקומי לריפו המרוחק.
- **`git clone <כתובת>`** — יוצר עותק מקומי חדש ומלא (כולל **כל ההיסטוריה**) מריפו מרוחק.
- **`git pull`** — מושך קומיטים חדשים מהריפו המרוחק אל הריפו המקומי (למשל, שינויים שדחף שותף).

```{note}
במחברת הזו נדגים עם תיקייה **מקומית** שמשמשת "עותק דמה" של GitHub (`--bare`), כדי שהדוגמה תרוץ באופן עצמאי בלי חיבור אינטרנט או פרטי התחברות. **אותן ארבע פקודות בדיוק** עובדות מול כתובת GitHub אמיתית — רק מחליפים את הנתיב המקומי בכתובת `https://github.com/...`.
```

### דוגמה: לגבות ריפו, ואז "לשחזר על מחשב חדש"

נבנה ריפו מקומי עם קומיט אחד, נדחוף אותו ל"GitHub", ואז נדמה מחשב חדש לגמרי (`clone` לתיקייה נפרדת) — ההוכחה שהגיבוי באמת עובד.

In [ ]:
import tempfile, os

base = tempfile.mkdtemp(prefix="git_github_")
origin_path = os.path.join(base, "origin_on_github")  # דמה של GitHub
local_path = os.path.join(base, "my_laptop")

os.makedirs(origin_path)
os.chdir(origin_path)
!git init -q --bare -b main   # ריפו "עירום" - בדיוק כמו ריפו ריק שיצרתם ב-GitHub

os.makedirs(local_path)
os.chdir(local_path)
!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
!git config color.ui false

In [ ]:
%%writefile script.py
print("שלום מהמעבדה")

In [ ]:
!git add script.py
!git commit -q -m "קומיט ראשון"
!git remote add origin {origin_path}
!git push -q -u origin main
!git log --oneline

`git remote add` קישר את הריפו המקומי ל"GitHub", ו-`git push` שלח את הקומיט לשם. עכשיו נדמה **מחשב חדש לגמרי** — תיקייה ריקה שמעולם לא ראתה את הפרויקט — ונשחזר ממנו:

In [ ]:
new_machine_path = os.path.join(base, "new_computer")
os.chdir(base)
!git clone -q {origin_path} {new_machine_path}

os.chdir(new_machine_path)
!git config color.ui false
!ls
!cat script.py
!git log --oneline

כל הקבצים **וכל ההיסטוריה** קיימים במחשב "החדש" — לא רק הקובץ האחרון, אלא כל הקומיטים. זו ההוכחה שהגיבוי עובד: `clone` לא מוריד "עותק", הוא משחזר את כל הריפו.

עכשיו נראה גם את הכיוון ההפוך — `pull`. נדמה ששותף למעבדה עובד על "המחשב החדש", מוסיף שינוי, ודוחף אותו. חוזרים ל"מחשב שלכם" (`my_laptop`) ומושכים את השינוי שלו:

In [ ]:
os.chdir(new_machine_path)
!git config user.email "partner@example.com"
!git config user.name "Partner"

In [ ]:
%%writefile -a script.py
print("שורה שהוסיף השותף")

In [ ]:
!git add script.py
!git commit -q -m "שותף: הוספת שורת הדפסה"
!git push -q origin main

In [ ]:
os.chdir(local_path)
!git pull -q origin main
!cat script.py
!git log --oneline

`my_laptop` — הריפו שבו התחלנו — עכשיו רואה את השינוי של השותף, בלי שהוא בכלל היה על אותו מחשב. זהו בדיוק תהליך העבודה עם GitHub בזוג: כל אחד `push`-ה את השינויים שלו, וכל אחד `pull`-ל את השינויים של האחר.

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "מה ההבדל בין git push לבין git commit?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "אין הבדל, שתי הפקודות עושות אותו דבר", "correct": False, "feedback": "לא — הן פועלות על שני ריפואים שונים לגמרי."},
            {"answer": "commit שומר תמונת מצב בריפו המקומי בלבד; push שולח קומיטים קיימים אל הריפו המרוחק", "correct": True, "feedback": "נכון."},
            {"answer": "push שומר תמונת מצב מקומית; commit שולח אותה ל-GitHub", "correct": False, "feedback": "זה הפוך — commit הוא המקומי, push הוא זה ששולח החוצה."},
            {"answer": "commit דורש חיבור אינטרנט, push לא", "correct": False, "feedback": "זה הפוך — commit עובד תמיד מקומית בלי אינטרנט; push הוא זה שדורש חיבור לריפו המרוחק."}
        ]
    },
    {
        "question": "מה ההבדל בין git clone לבין git pull?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "clone יוצר עותק מקומי חדש ומלא של ריפו שעדיין אין לכם; pull מעדכן ריפו מקומי שכבר קיים אצלכם עם קומיטים חדשים מהמרוחק", "correct": True, "feedback": "נכון."},
            {"answer": "pull יוצר עותק חדש; clone מעדכן עותק קיים", "correct": False, "feedback": "זה הפוך."},
            {"answer": "שתיהן זהות, רק שמות שונים לאותה פעולה", "correct": False, "feedback": "לא — משתמשים בהן במצבים שונים לגמרי, כמו שהוסבר למעלה."},
            {"answer": "clone עובד רק פעם אחת בחיי הריפו, pull אפשר להריץ רק פעם אחת ביום", "correct": False, "feedback": "אין הגבלות כאלה על אף אחת מהפקודות."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

תרגלו את מעגל ה-push/pull המלא בעצמכם: צרו ריפו-דמה נוסף (`origin2`), ריפו מקומי (`laptop_a`) שדוחף אליו קומיט ראשון, ואז ריפו שני (`laptop_b`, שמתקבל דרך `clone` מ-`origin2`) שמוסיף קומיט ודוחף אותו בחזרה. לבסוף, חזרו ל-`laptop_a` ומשכו (`pull`) את הקומיט של `laptop_b`.

In [ ]:
# base2 = tempfile.mkdtemp(prefix="git_practice_gh_")
# origin2 = ...
# laptop_a = ...
# laptop_b = ... (נוצר עם git clone)
#
# כתבו כאן את כל הרצף

`````{admonition} פתרון
:class: dropdown, tip
```python
base2 = tempfile.mkdtemp(prefix="git_practice_gh_")
origin2 = os.path.join(base2, "origin2")
laptop_a = os.path.join(base2, "laptop_a")

os.makedirs(origin2); os.chdir(origin2)
!git init -q --bare -b main

os.makedirs(laptop_a); os.chdir(laptop_a)
!git init -q -b main
!git config user.email "a@example.com"; !git config user.name "A"; !git config color.ui false
```
```python
%%writefile notes.txt
קומיט ראשון מ-laptop_a
```
```python
!git add notes.txt
!git commit -q -m "קומיט ראשון"
!git remote add origin {origin2}
!git push -q -u origin main
```
```python
laptop_b = os.path.join(base2, "laptop_b")
os.chdir(base2)
!git clone -q {origin2} {laptop_b}
os.chdir(laptop_b)
!git config user.email "b@example.com"; !git config user.name "B"; !git config color.ui false
```
```python
%%writefile -a notes.txt
שורה מ-laptop_b
```
```python
!git add notes.txt
!git commit -q -m "laptop_b: הוספת שורה"
!git push -q origin main
```
```python
os.chdir(laptop_a)
!git pull -q origin main
!cat notes.txt
```
`````

### לבית: הגיבוי האמיתי ל-GitHub

התרגול האמיתי הוא מול GitHub עצמו, לא ריפו-דמה מקומי. לביצוע (מחוץ למחברת, בטרמינל אמיתי או VS Code):

```bash
# 1. צרו repo אישי ריק חדש באתר github.com (בלי README, בלי .gitignore - ריק לגמרי)
# 2. בתוך תיקיית התרגול שלכם מסעיפים 13.2-13.3:
git remote add origin https://github.com/<username>/<repo-name>.git
git push -u origin main

# 3. בדקו שהגיבוי אמיתי: שכפלו לתיקייה נקייה לגמרי
git clone https://github.com/<username>/<repo-name>.git test_clone
```